# LanceDB fundamentals

LanceDB is an open-source vector database designed to handle large-scale vector data efficiently. It provides a robust platform for storing, indexing, and querying high-dimensional vectors, making it ideal for applications in machine learning, computer vision, and natural language processing.

We'll be using LanceDB for the purpose of creating RAG applications. However before we dive into that, let's cover some fundamental concepts of LanceDB, so that you know what is happening under the hood.

## How to read this notebook (map)

**What you will learn, in order**

| Part | Cells | Idea |
| --- | --- | --- |
| 1. Connect | `lancedb.connect` | A LanceDB database is just a folder |
| 2. Tables | create / add / open / drop | A table = rows with a `text` and a `vector` column |
| 3. Vector search | `search(vector)` | "Find the rows nearest to this list of numbers" |
| 4. Embeddings API | `get_registry().get("cohere")` | Let LanceDB turn text into vectors for you |
| 5. Text search | `search("some text")` | Same thing, but you type words, not numbers |
| 6. Hybrid search | `create_fts_index` + `query_type="hybrid"` | Keyword search + meaning search together |

**Files this notebook touches**

| File / folder | Role | Direction |
| --- | --- | --- |
| `animals_text_embeddings.json` | 6 animal sentences with hand-made 3-number vectors | read |
| `jokes.json` | 30 short jokes (`{"jokes": [...]}`) | read |
| `vector_database/` | the database folder; created by this notebook | written |
| `.env` -> `COHERE_API_KEY` | key for the Cohere embedding model (cells in part 4 onwards) | read |

This notebook does **not** use `custom_embeddings/` or `lancedb_embeddings.ipynb`. They are the next step.

## Connect LanceDB

- when connecting to LanceDB it will create a folder for your lancedb database if it does not already exist

In [2]:
# STEP 1 - open (or create) the database.
# `uri` is just a FOLDER path. If "vector_database/" does not exist, LanceDB creates it.
# There is no server to start: LanceDB is "embedded", like SQLite or DuckDB.
# The returned `db` object is the handle we use for everything else (create/open/drop tables).
import lancedb

db = lancedb.connect(uri = "vector_database")
db

LanceDBConnection(uri='c:\\Users\\adelo\\de25\\ai_ops_lecture\\09_lancedb\\vector_database')

In [3]:
# The absolute path of the folder where all our tables are stored.
db.uri

'c:\\Users\\adelo\\de25\\ai_ops_lecture\\09_lancedb\\vector_database'

## Create a Table

when creating a table, it is stored as a folder with .lance suffix containing the following folders
- _transactions
- _versions
- data

a table is a pyarraow table stored in the data folder

In [4]:
# STEP 2 - load some example rows from a local JSON file.
# Each item in the list is ONE ROW: {"text": <sentence>, "vector": [3 numbers]}.
# The 3 numbers are a hand-made "fake embedding" so we can see how vector search works
# without calling any AI model. Real embeddings have hundreds of numbers (see later cells).
import json
with open("animals_text_embeddings.json") as file:
    data = json.loads(file.read())

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [5]:
# Create the table "animals_text" from our list of dicts.
# - LanceDB guesses the columns (schema) from the data: text -> string, vector -> list of 3 floats.
# - exist_ok=True: IDEMPOTENT - if the table already exists, open it instead of crashing (safe to re-run the cell).
# - The result shows version=1: LanceDB keeps a version number for every change to a table.
table_animals = db.create_table("animals_text", exist_ok=True, data=data)
table_animals


LanceTable(name='animals_text', _conn=LanceDBConnection(uri='c:\\Users\\adelo\\de25\\ai_ops_lecture\\09_lancedb\\vector_database'))

In [6]:
# .head() returns the first rows as a pyarrow Table (the columnar format LanceDB stores data in).
# Default is 5 rows, which is why the 6th animal (frog) is not shown.
# it's a pyarrow table 
table_animals.head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

In [7]:
# Same data, but as a pandas DataFrame - easier to read in a notebook.
# converting to pandas 
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


add some more data to the table

In [8]:
# .add() APPENDS new rows to an existing table. Each add() creates a new version (now version=2).
# The new rows must have the same columns as the table (text + a 3-number vector).
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

table_animals.add(more_data)

AddResult(version=5)

In [9]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Create an empty table and then delete it

In [10]:
# STEP 3 - create an EMPTY table by describing its columns instead of passing data.
# LanceModel is a Pydantic model (same idea as lectures 03/04) that LanceDB turns into a table schema.
# Here: a table with a text column `joke` and an integer column `rating`. No rows yet.
# Note: this is a throw-away demo table; the real "jokes" table is created further down.
# LanceModel is a Pydantic model that can be converted to LanceDB table
from lancedb.pydantic import LanceModel

class JokeSchema(LanceModel):
    joke: str 
    rating: int


# mode="overwrite": if a table called "jokes" is already in the folder (e.g. from an earlier run of this notebook,
# which has different columns), replace it. With exist_ok=True LanceDB would instead raise
# "Provided schema does not match existing table schema".
db.create_table(name = "jokes",schema=JokeSchema, mode="overwrite")

LanceTable(name='jokes', _conn=LanceDBConnection(uri='c:\\Users\\adelo\\de25\\ai_ops_lecture\\09_lancedb\\vector_database'))

In [11]:
# List the tables in this database. (DeprecationWarning in the output: newer LanceDB prefers
# db.list_tables(). Both work for now.)
db.table_names()

C:\Users\adelo\AppData\Local\Temp\ipykernel_11680\3313488180.py:3: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  db.table_names()


['animals_text', 'jokes']

In [12]:
# Delete the demo "jokes" table (removes its .lance folder). Careful: this cannot be undone.
db.drop_table("jokes")

In [13]:
# Check again: only "animals_text" is left.
db.table_names()

C:\Users\adelo\AppData\Local\Temp\ipykernel_11680\1098963777.py:2: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  db.table_names()


['animals_text']

## Open existing table

In [14]:
# Re-open a table that already exists in this database by its name.
# This is how you come back to your data in a new session: connect() then open_table().
# since db is already connected to vector_database it has animals_text as table
db.open_table("animals_text").head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

## Vector search in LanceDB

Uses ANN - approximate nearest neighbours for searching vectors

- search with vector directly
- note that the vector column is already an example embedding simulated with LLM

A more realistic approach is to just have text records, use an existing embedding model and embedd those text records, then embed the query vector and find closest match 


In [15]:
# Look at the data again before searching. Each text has a 3-number vector next to it.
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [16]:
# STEP 4 - vector search with a raw vector.
# `query_vector` must have the same length (3) as the vectors in the table.
# search() finds the rows whose vector is NEAREST to the query vector.
# `_distance` is added by LanceDB: SMALLER = CLOSER. It is the squared Euclidean distance, e.g. for the panda:
#   (0.51-0.5)^2 + (0.37-0.2)^2 + (0.82-0.9)^2 = 0.0001 + 0.0289 + 0.0064 = 0.0354
# .limit(3) keeps only the 3 closest rows (top-k).
query_vector = [0.5, 0.2, 0.9]

# finds closest rows based on the Euclidean distance by default
table_animals.search(query_vector).limit(3).to_pandas()

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
2,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354


## Embeddings API (Google Gemini)


In [35]:
# LanceDB's Gemini embedder ONLY looks for an environment variable called GOOGLE_API_KEY.
# A key from Google AI Studio (what you call your Gemini key) is the same kind of key, it just has another name.
# So: load .env, then copy GEMINI_API_KEY into the name LanceDB expects. Run this BEFORE .create(...).
import os
from dotenv import load_dotenv

load_dotenv()  # reads the .env file into the environment
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]  # KeyError here = GEMINI_API_KEY is not in your .env

from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry

model = get_registry().get("gemini-text").create(name="gemini-embedding-001")
model


GeminiText(max_retries=7, name='gemini-embedding-001', dim=None, query_task_type='retrieval_query', source_task_type='retrieval_document')

In [36]:
# NOTE: we pass a STRING, and Python treats a string as a sequence of letters, so we get 5 vectors (h, e, l, l, o).
# To embed the whole word, pass a LIST: model.generate_embeddings(["hello"])  ->  1 vector.
model.generate_embeddings("hello")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


[array([-1.08947430e-02, -2.84894180e-03,  8.93425000e-03, -5.45415130e-02,
        -7.43374700e-03,  1.75042060e-02,  1.46537490e-02,  2.49882300e-02,
         1.15361300e-02, -4.03161900e-03,  1.96894370e-03, -1.58960150e-02,
        -3.36537770e-03,  1.59759690e-02,  9.17760600e-02, -2.41352900e-02,
         2.66211630e-02, -1.59987620e-02, -9.19349400e-03, -3.23178250e-03,
         1.73149180e-02, -1.41047070e-02, -5.99844000e-04, -1.38508780e-02,
         2.58160250e-02,  2.71950230e-02,  1.77555960e-02,  9.80010800e-03,
         2.23747080e-02,  1.73749590e-02, -2.76534210e-03,  6.73423240e-03,
         4.06015020e-04,  2.77890300e-03, -4.35200500e-03,  1.15051350e-02,
         2.28939220e-02, -1.01079650e-02, -1.30854435e-02,  2.23838240e-02,
         3.03857260e-04, -7.13414700e-03,  2.23933700e-03, -1.29814250e-03,
         7.04156700e-05,  1.10823490e-02, -5.63707800e-03, -2.57279570e-02,
         5.90762540e-03,  4.22959360e-03,  1.75436330e-03,  1.19375810e-02,
        -2.0

In [40]:
# Shape is (5, 768): 5 vectors (one per letter of "hello") x 768 numbers each (Gemini's vector length).
# With a list, np.array(model.generate_embeddings(["hello"])).shape would be (1, 768).
import numpy as np
hello_embedding = np.array(model.generate_embeddings("hello"))
hello_embedding.shape

(5, 768)

## Embeddings API (Cohere)

LanceDB provides an Embeddings API that allows you to easily generate and store vector embeddings for your data. This is particularly useful for applications such as semantic search, recommendation systems, and clustering.

We'll use an embeddings model from Cohere here (a hosted API; needs `COHERE_API_KEY`). The next notebook uses a free local sentence-transformers model instead

**Data flow in the second half of this notebook**

```
jokes.json -> pandas DataFrame -> table.add() -> [Cohere embeds each joke] -> table "jokes" (joke + vector)
                                                                                   |
 "give me nature related jokes" -> [Cohere embeds the query] -> nearest vectors ---+--> results
                                                                keyword index (FTS) --+  (hybrid: both, merged by RRF)
```

In [19]:
# STEP 5 - let LanceDB create the embeddings for us (the "Embeddings API").
# This cell uses COHERE's hosted model. It needs COHERE_API_KEY in your environment/.env and calls the internet.
# The registry is a catalogue of embedding providers: get("cohere") picks one, create(name=...) configures it.
# This model turns any text into a list of 384 numbers (model.ndims() == 384 in our run; check yours).
from dotenv import load_dotenv
load_dotenv()  # loads COHERE_API_KEY from .env. Without the key, table.add() below fails with "Could not find API key for cohere"

from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry


model = get_registry().get("cohere").create(name = "embed-multilingual-light-v3.0")


class JokeModel(LanceModel):
    joke: str = model.SourceField() # marks text as input to embedding function
    # ^ SOURCE = the text you give LanceDB. It is sent to the embedding model automatically.
    vector: Vector(model.ndims()) = model.VectorField() # vector is destination of computed embedding
    # ^ DESTINATION = the numbers come back here. We never fill this column ourselves.

# Create the table from the schema above. It is empty for now; embeddings are made when we add rows.
table_jokes = db.create_table("jokes", schema = JokeModel, exist_ok=True)
table_jokes



LanceTable(name='jokes', _conn=LanceDBConnection(uri='c:\\Users\\adelo\\de25\\ai_ops_lecture\\09_lancedb\\vector_database'))

In [20]:
# Load the 30 jokes. jokes.json looks like {"jokes": ["...", "..."]}, so pandas makes ONE column named "jokes".
# Our table's text column is called `joke` (singular), so we rename it to match the schema.
# (Compare with lancedb_embeddings.ipynb, where the column is called `jokes` and no rename is needed.)
import pandas as pd 
# encoding="utf-8" is needed on Windows: jokes.json is UTF-8 (curly quotes) but Windows opens files as cp1252 by default,
# which crashes with UnicodeDecodeError.
with open("jokes.json", "r", encoding="utf-8") as file:
    jokes_data = json.loads(file.read())

df_jokes = pd.DataFrame(jokes_data).rename({"jokes": "joke"}, axis=1)
df_jokes.head()

,joke
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


adds data to our table

In [21]:
# Add the 30 jokes. We only provide the `joke` column: LanceDB sends each text to Cohere, gets back
# 384 numbers, and stores them in the `vector` column for us. (One API call batch = uses your Cohere quota.)


table_jokes.add(df_jokes)


AddResult(version=2)

we can see that vector column has been created automatically and embeddings have been generated for each joke

In [22]:
# The `vector` column now exists and is filled: 384 numbers per joke.
table_jokes.head(2)

pyarrow.Table
joke: string not null
vector: fixed_size_list<item: float>[384]
  child 0, item: float
----
joke: [["Parallel lines have so much in common—it’s sad they’ll never meet.","ETL stands for “Extract, Transform, Leave for the next person.”"]]
vector: [[[-0.019195557,0.016357422,0.003129959,0.021896362,0.123168945,...,-0.06878662,0.009521484,-0.013084412,0.103637695,0.013320923],[0.03878784,0.026245117,-0.062469482,0.007419586,-0.0026416779,...,0.032409668,-0.078186035,-0.03564453,0.032104492,0.050231934]]]

In [23]:
table_jokes.to_pandas().head()

,joke,vector
0,Parallel lines have so much in common—it’s sad...,"[-0.019196, 0.016357, 0.00313, 0.021896, 0.123..."
1,"ETL stands for “Extract, Transform, Leave for ...","[0.038788, 0.026245, -0.062469, 0.00742, -0.00..."
2,What do you call a snake that runs your script...,"[0.062378, 0.039581, 0.043427, -0.026855, -0.0..."
3,"Gold walks into a bar. The bartender says, “Au...","[-0.015686, -0.051147, -0.066101, 0.034607, 0...."
4,C# devs don’t argue; they just throw exceptions.,"[-0.036835, 0.081909, -0.012024, -0.044586, -0..."


## Vector Search

In [24]:
# STEP 6 - search with TEXT. LanceDB embeds the query with the SAME model, then finds the nearest jokes.
# That is why "data engineering jokes" finds jokes about pipelines and data lakes: the MEANING is close,
# even if the words are different. Results are sorted by _distance (smaller = closer).
table_jokes.search("data engineering jokes").limit(8).to_pandas()

,joke,vector,_distance
0,Why do data engineers hate nature? Too many un...,"[0.068481, 0.057648, -0.085327, -0.04184, 0.04...",1.181525
1,"Data engineer motto: If it works, don’t touch ...","[-0.08551, -0.036652, -0.03038, -0.02034, 0.00...",1.221188
2,The C# compiler walked into a bar. The bartend...,"[-0.043457, -0.015358, -0.045044, 0.039612, 0....",1.459619
3,I told a chemistry joke… there was no reaction.,"[0.042938, 0.079468, -0.019989, 0.003523, 0.11...",1.501731
4,I asked the data lake if it had my file. It sa...,"[0.017838, 0.007957, -0.010849, -0.01738, 0.08...",1.527514
5,C# programmers love coffee—they're always work...,"[-0.099854, 0.021042, -0.024811, 0.0383, -0.00...",1.528505
6,"Our data pipeline broke… but don't worry, it w...","[-0.046173, -0.003428, 0.050476, -0.023209, 0....",1.561503
7,"Gold walks into a bar. The bartender says, “Au...","[-0.015686, -0.051147, -0.066101, 0.034607, 0....",1.657165


In [25]:
# Same search, different topic. Notice that jokes about Python also show up for "C#":
# they are about programming, so they are 'near' in meaning, just further away.
table_jokes.search("give me jokes on C#").limit(8).to_pandas()

,joke,vector,_distance
0,The C# compiler walked into a bar. The bartend...,"[-0.043457, -0.015358, -0.045044, 0.039612, 0....",0.995173
1,Why is C# like a musical? It has so many classes.,"[-0.021835, 0.045624, -0.001379, -0.050201, 0....",1.033812
2,C# devs don’t argue; they just throw exceptions.,"[-0.036835, 0.081909, -0.012024, -0.044586, -0...",1.129056
3,C# programmers love coffee—they're always work...,"[-0.099854, 0.021042, -0.024811, 0.0383, -0.00...",1.186593
4,Why did the C# developer go broke? He kept usi...,"[-0.025772, 0.061371, 0.003834, 0.030746, -0.0...",1.322273
5,I told a chemistry joke… there was no reaction.,"[0.042938, 0.079468, -0.019989, 0.003523, 0.11...",1.375587
6,My math teacher said I’m average… how mean!,"[-0.073547, 0.087524, -0.067993, -0.041931, 0....",1.523852
7,I tried to explain async/await to my friend… n...,"[-0.089233, 0.002813, -0.002365, 0.009552, 0.0...",1.526051


In [26]:
# Another topic. We always get 8 rows back (limit), even when only a few are really about chemistry.
# Vector search never says "no match"; it always returns the NEAREST rows.
table_jokes.search("give me chemistry related jokes").limit(8).to_pandas()

,joke,vector,_distance
0,I told a chemistry joke… there was no reaction.,"[0.042938, 0.079468, -0.019989, 0.003523, 0.11...",0.951342
1,Oxygen and Magnesium got together—OMg!,"[0.034821, 0.027802, -0.066833, -0.019485, 0.1...",1.161264
2,Chemists have all the solutions… mostly in bea...,"[-0.002092, 0.000159, -0.047211, -0.009438, 0....",1.230715
3,Why did the chemist ground his kids? Because t...,"[-0.067444, 0.000148, -0.037079, -0.041199, 0....",1.293402
4,Never trust an atom—they make up everything.,"[0.047852, 0.006481, -0.053406, -0.052246, 0.0...",1.402527
5,The C# compiler walked into a bar. The bartend...,"[-0.043457, -0.015358, -0.045044, 0.039612, 0....",1.428939
6,"Gold walks into a bar. The bartender says, “Au...","[-0.015686, -0.051147, -0.066101, 0.034607, 0....",1.489073
7,I put “root beer” in a square glass… now it’s ...,"[0.007595, 0.032043, 0.042328, -0.039154, 0.04...",1.515764


In [27]:
# "nature" is vague: no joke is really about nature. Look how loosely related the results are.
# This is the weakness of pure vector search that hybrid search (below) tries to improve.
table_jokes.search("give me nature related jokes").limit(8).to_pandas()

,joke,vector,_distance
0,I told a chemistry joke… there was no reaction.,"[0.042938, 0.079468, -0.019989, 0.003523, 0.11...",1.455355
1,Why do data engineers hate nature? Too many un...,"[0.068481, 0.057648, -0.085327, -0.04184, 0.04...",1.527386
2,Oxygen and Magnesium got together—OMg!,"[0.034821, 0.027802, -0.066833, -0.019485, 0.1...",1.649310
3,I asked the data lake if it had my file. It sa...,"[0.017838, 0.007957, -0.010849, -0.01738, 0.08...",1.689583
4,Why did the Python programmer get bitten? Beca...,"[0.028275, 0.053619, 0.034119, 0.073059, 0.024...",1.704022
5,I put “root beer” in a square glass… now it’s ...,"[0.007595, 0.032043, 0.042328, -0.039154, 0.04...",1.705119
6,The C# compiler walked into a bar. The bartend...,"[-0.043457, -0.015358, -0.045044, 0.039612, 0....",1.717599
7,"Gold walks into a bar. The bartender says, “Au...","[-0.015686, -0.051147, -0.066101, 0.034607, 0....",1.726934


## Hybrid search

hybrid search combines traditional keyword-based search with vector-based search to provide more relevant results. This approach leverages the strengths of both methods, allowing for more accurate and context-aware retrieval of information.

In [28]:
# STEP 7 - hybrid search. First build a FULL-TEXT SEARCH (FTS) index on the `joke` column.
# FTS = classic keyword search (words are indexed, like the index at the back of a book).
# replace=True: rebuild the index if it already exists.
# to enable keyword based search on joke column
table_jokes.create_fts_index("joke", replace=True)


C:\Users\adelo\AppData\Local\Temp\ipykernel_11680\292266442.py:5: DeprecatedWarning: create_fts_index is deprecated as of 0.25.0. Use create_index() with config=FTS() instead.
  table_jokes.create_fts_index("joke", replace=True)


In [29]:
# A reranker merges two ranked lists into one. RRF = Reciprocal Rank Fusion:
# each row gets a score from its RANK in each list (rank 1 counts most), and the scores are added.
# Hybrid = vector search (meaning) + keyword search (exact words), blended into ONE ranking.
from lancedb import rerankers

# reranks the results from a vector and full text search
reranker = rerankers.RRFReranker()

results = table_jokes.search(
    "give me nature related jokes",
    query_type="hybrid",       # run vector search AND keyword (FTS) search, then merge
    vector_column_name="vector",  # which column holds the embeddings
    fts_columns="joke",           # which text column was indexed for keywords
).rerank(reranker).limit(8).to_pandas()  # merge the two lists with RRF, keep the top 8

# somewhat different to before
results


,joke,vector,_relevance_score
0,I told a chemistry joke… there was no reaction.,"[0.042938, 0.079468, -0.019989, 0.003523, 0.11...",0.032787
1,Why do data engineers hate nature? Too many un...,"[0.068481, 0.057648, -0.085327, -0.04184, 0.04...",0.032258
2,Oxygen and Magnesium got together—OMg!,"[0.034821, 0.027802, -0.066833, -0.019485, 0.1...",0.015873
3,I asked the data lake if it had my file. It sa...,"[0.017838, 0.007957, -0.010849, -0.01738, 0.08...",0.015625
4,Why did the Python programmer get bitten? Beca...,"[0.028275, 0.053619, 0.034119, 0.073059, 0.024...",0.015385
5,I put “root beer” in a square glass… now it’s ...,"[0.007595, 0.032043, 0.042328, -0.039154, 0.04...",0.015152
6,The C# compiler walked into a bar. The bartend...,"[-0.043457, -0.015358, -0.045044, 0.039612, 0....",0.014925
7,"Gold walks into a bar. The bartender says, “Au...","[-0.015686, -0.051147, -0.066101, 0.034607, 0....",0.014706


## Rule of thumb 

- For exact matching -> FTS
- For meaning based matching -> Vector search
- Unpredictable or mixed queries -> Hybrid search
